# ♟️ chesSon: 100K+ Games Training (v2)

This notebook trains **chesSon** using multiple Kaggle datasets simultaneously.

### What's New in v2:
1. **Multi-format support**: Reads `.pgn.zst`, `.pgn`, and `.csv` files automatically.
2. **Fine-tuning mode**: Loads your existing `chesson_weights.pt` and continues training.
3. **Elo filtering**: Only trains on games above a minimum rating threshold.
4. **Multi-source**: Combine multiple Kaggle datasets for maximum game count.
5. **FP16 Mixed Precision**: Maximizes throughput on Kaggle GPUs (T4 / P100).

### Recommended Kaggle Datasets to Add:
| Dataset | Format | ~Games |
|---|---|---|
| [datasnaek/chess](https://www.kaggle.com/datasets/datasnaek/chess) | CSV | 20,000 |
| [lichess-open-database monthly .pgn.zst](https://database.lichess.org/) | .pgn.zst | 100M+ |
| [arevel/chess-games](https://www.kaggle.com/datasets/arevel/chess-games) | CSV | 6.25M |
| [milesh1/35-million-chess-games](https://www.kaggle.com/datasets/milesh1/35-million-chess-games) | TXT | 35M |

In [ ]:
# Step 1: Install required libraries
!pip install python-chess zstandard tqdm pandas -q
print("Installed dependencies successfully!")

In [ ]:
# Step 2: Check what datasets are available
import glob, os

print("=== Available input files ===")
for ext in ['*.pgn.zst', '*.pgn', '*.csv']:
    files = glob.glob(f'/kaggle/input/**/{ext}', recursive=True)
    for f in files:
        size_mb = os.path.getsize(f) / 1e6
        print(f"  {ext.split('.')[-1].upper():6s}  {size_mb:8.1f} MB  {f}")

print()
print("=== GPU available ===")
import torch
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Step 3: Imports & Global Constants
import io, os, sys, time, glob
from typing import Iterator, Optional, Tuple

import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from tqdm.notebook import tqdm
import zstandard as zstd
import pandas as pd

ACTION_SIZE = 4168
NUM_PLANES = 14
PIECE_TO_PLANE = {
    chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
    chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5,
}
print("Imports OK")

In [ ]:
# Step 4: Configuration — EDIT THIS CELL
CONFIG = {
    # ---- Data sources (all formats supported) ----
    # Add or remove paths based on what you've attached as Kaggle datasets
    "data_sources": [
        "/kaggle/input/**/*.pgn.zst",   # Lichess .pgn.zst files
        "/kaggle/input/**/*.pgn",       # Any plain PGN files
        "/kaggle/input/**/*.csv",       # CSV datasets (datasnaek/chess, arevel, etc.)
    ],

    # ---- Fine-tuning (set to None to train from scratch) ----
    # Upload your existing chesson_weights.pt as a Kaggle dataset and point here:
    "pretrained_weights": "/kaggle/input/chesson-weights/chesson_weights.pt",

    # ---- Training hyperparameters ----
    "batch_size": 512,
    "lr": 3e-4,           # Use 1e-3 for scratch training, 3e-4 for fine-tuning
    "weight_decay": 1e-4,
    "num_res_blocks": 6,
    "channels": 128,
    "max_positions": 3_000_000,   # ~100K games × 30 moves average
    "min_moves_per_game": 15,
    "elo_filter": 1500,           # Only use games where both players >= 1500 Elo

    # ---- Output ----
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "output_weights": "/kaggle/working/chesson_weights.pt",
    "log_interval": 100,
}

print(f"Device: {CONFIG['device']}")
print(f"Max positions: {CONFIG['max_positions']:,}")
print(f"Elo filter: {CONFIG['elo_filter']}")
print(f"Fine-tuning from: {CONFIG['pretrained_weights']}")

In [ ]:
# Step 5: Board & Move Encoding
def encode_board(board: chess.Board) -> np.ndarray:
    planes = np.zeros((NUM_PLANES, 8, 8), dtype=np.float32)
    canonical = board.mirror() if board.turn == chess.BLACK else board
    for square in chess.SQUARES:
        piece = canonical.piece_at(square)
        if piece is not None:
            rank = chess.square_rank(square)
            file = chess.square_file(square)
            plane_idx = PIECE_TO_PLANE[piece.piece_type] + (0 if piece.color == chess.WHITE else 6)
            planes[plane_idx, rank, file] = 1.0
    if canonical.has_kingside_castling_rights(chess.WHITE):  planes[12, 0, 4] = planes[12, 0, 7] = 1.0
    if canonical.has_queenside_castling_rights(chess.WHITE): planes[12, 0, 4] = planes[12, 0, 0] = 1.0
    if canonical.has_kingside_castling_rights(chess.BLACK):  planes[12, 7, 4] = planes[12, 7, 7] = 1.0
    if canonical.has_queenside_castling_rights(chess.BLACK): planes[12, 7, 4] = planes[12, 7, 0] = 1.0
    if canonical.ep_square is not None:
        planes[13, chess.square_rank(canonical.ep_square), chess.square_file(canonical.ep_square)] = 1.0
    return planes

def move_to_index(move: chess.Move, turn: chess.Color) -> int:
    if turn == chess.BLACK:
        from_sq = chess.square_mirror(move.from_square)
        to_sq   = chess.square_mirror(move.to_square)
    else:
        from_sq, to_sq = move.from_square, move.to_square
    if move.promotion is not None and move.promotion != chess.QUEEN:
        slot = (from_sq - 48) * 3 + (chess.square_file(to_sq) - chess.square_file(from_sq) + 1)
        return 4096 + {chess.KNIGHT: 0, chess.BISHOP: 24, chess.ROOK: 48}[move.promotion] + slot
    return from_sq * 64 + to_sq

print("Encoding functions defined.")

In [ ]:
# Step 6: Multi-Source Streaming Dataset
def _result_to_value(result, is_white_turn):
    if result == '1-0':     return 1.0 if is_white_turn else -1.0
    if result == '0-1':     return -1.0 if is_white_turn else 1.0
    if result == '1/2-1/2': return 0.0
    return None

def _yield_from_game(game, result, min_moves, elo_filter):
    moves = list(game.mainline_moves())
    if len(moves) < min_moves: return
    if elo_filter > 0:
        try:
            if (int(game.headers.get('WhiteElo','0') or '0') < elo_filter or
                int(game.headers.get('BlackElo','0') or '0') < elo_filter):
                return
        except: pass
    board = game.board()
    for move in moves:
        if not board.is_legal(move): break
        val = _result_to_value(result, board.turn == chess.WHITE)
        if val is not None:
            yield encode_board(board), move_to_index(move, board.turn), np.float32(val)
        board.push(move)

def stream_pgn_zst(path, min_moves, elo_filter):
    dctx = zstd.ZstdDecompressor()
    with open(path, 'rb') as fh:
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')
            while True:
                try:   game = chess.pgn.read_game(text_stream)
                except: continue
                if game is None: break
                result = game.headers.get('Result', '*')
                if result in ('1-0', '0-1', '1/2-1/2'):
                    yield from _yield_from_game(game, result, min_moves, elo_filter)

def stream_pgn(path, min_moves, elo_filter):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        while True:
            try:   game = chess.pgn.read_game(f)
            except: continue
            if game is None: break
            result = game.headers.get('Result', '*')
            if result in ('1-0', '0-1', '1/2-1/2'):
                yield from _yield_from_game(game, result, min_moves, elo_filter)

def stream_csv(path, min_moves, elo_filter):
    df = pd.read_csv(path)
    move_col = next((c for c in ['moves','pgn','game','uci'] if c in df.columns), None)
    if move_col is None:
        print(f'[SKIP] No moves column in {path}'); return
    for _, row in df.iterrows():
        if 'winner' in df.columns:
            result = {'white':'1-0','black':'0-1','draw':'1/2-1/2'}.get(str(row['winner']).lower())
        elif 'result' in df.columns:
            result = row['result'] if row['result'] in ('1-0','0-1','1/2-1/2') else None
        else: result = None
        if result is None: continue
        if elo_filter > 0:
            wr = int(row.get('white_rating', row.get('WhiteElo', 0)) or 0)
            br = int(row.get('black_rating', row.get('BlackElo', 0)) or 0)
            if (wr > 0 and wr < elo_filter) or (br > 0 and br < elo_filter): continue
        board = chess.Board()
        parsed = []
        for tok in str(row[move_col]).split():
            try:
                m = chess.Move.from_uci(tok)
                if board.is_legal(m): parsed.append(m); board.push(m)
                else: break
            except:
                try: m = board.parse_san(tok); parsed.append(m); board.push(m)
                except: break
        if len(parsed) < min_moves: continue
        board = chess.Board()
        for m in parsed:
            val = _result_to_value(result, board.turn == chess.WHITE)
            if val is not None:
                yield encode_board(board), move_to_index(m, board.turn), np.float32(val)
            board.push(m)

class MultiSourceChessDataset(IterableDataset):
    def __init__(self, data_sources, max_positions, min_moves, elo_filter):
        self.max_positions = max_positions
        self.min_moves = min_moves
        self.elo_filter = elo_filter
        self.files = []
        for src in data_sources:
            if '*' in src or '?' in src: self.files.extend(sorted(glob.glob(src, recursive=True)))
            elif os.path.isfile(src): self.files.append(src)
        seen = set()
        self.files = [f for f in self.files if not (f in seen or seen.add(f))]
        print(f'Found {len(self.files)} file(s):')
        for f in self.files: print(f'  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)')

    def _stream_file(self, path):
        if path.endswith('.pgn.zst'): yield from stream_pgn_zst(path, self.min_moves, self.elo_filter)
        elif path.endswith('.pgn'):   yield from stream_pgn(path, self.min_moves, self.elo_filter)
        elif path.endswith('.csv'):   yield from stream_csv(path, self.min_moves, self.elo_filter)

    def __iter__(self):
        count = 0
        for path in self.files:
            print(f'\nStreaming: {os.path.basename(path)}')
            for sample in self._stream_file(path):
                yield sample
                count += 1
                if count >= self.max_positions: return

print('Dataset classes defined.')

In [ ]:
# Step 7: Model Definition (identical to chesson/model.py)
class ResidualBlock(nn.Module):
    def __init__(self, channels=128):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.GroupNorm(4, channels), nn.SiLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.GroupNorm(4, channels),
        )
        self.act = nn.SiLU(inplace=True)
    def forward(self, x): return self.act(x + self.block(x))

class ChesSonNet(nn.Module):
    def __init__(self, num_blocks=6, channels=128):
        super().__init__()
        self.input = nn.Sequential(
            nn.Conv2d(NUM_PLANES, channels, 3, padding=1, bias=False),
            nn.GroupNorm(4, channels), nn.SiLU(inplace=True),
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(channels) for _ in range(num_blocks)])
        self.policy = nn.Sequential(
            nn.Conv2d(channels, 64, 1, bias=False), nn.GroupNorm(4, 64), nn.SiLU(inplace=True),
            nn.Flatten(), nn.Linear(64*8*8, ACTION_SIZE),
        )
        self.value = nn.Sequential(
            nn.Conv2d(channels, 32, 1, bias=False), nn.GroupNorm(4, 32), nn.SiLU(inplace=True),
            nn.Flatten(), nn.Linear(32*8*8, 128), nn.SiLU(inplace=True), nn.Linear(128, 1), nn.Tanh(),
        )
    def forward(self, x):
        x = self.input(x)
        x = self.res_blocks(x)
        return self.policy(x), self.value(x)

model = ChesSonNet(num_blocks=CONFIG['num_res_blocks'], channels=CONFIG['channels'])
print(f"ChesSonNet: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters")

In [ ]:
# Step 8: Load pretrained weights for fine-tuning
device = torch.device(CONFIG['device'])
model = model.to(device)

pretrained = CONFIG.get('pretrained_weights')
if pretrained and os.path.isfile(pretrained):
    state = torch.load(pretrained, map_location=device)
    model.load_state_dict(state, strict=True)
    print(f'✅ Loaded pretrained weights from: {pretrained}')
    print('   Mode: FINE-TUNING (continuing from existing checkpoint)')
else:
    print('ℹ️  No pretrained weights found — training from scratch.')
    print(f'   (Looked for: {pretrained})')

In [ ]:
# Step 9: Training!
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
policy_criterion = nn.CrossEntropyLoss()
value_criterion  = nn.MSELoss()

dataset = MultiSourceChessDataset(
    data_sources=CONFIG['data_sources'],
    max_positions=CONFIG['max_positions'],
    min_moves=CONFIG['min_moves_per_game'],
    elo_filter=CONFIG.get('elo_filter', 0),
)
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], num_workers=2, pin_memory=True)

model.train()
total_loss, correct_top1, correct_top3, total_samples = 0.0, 0, 0, 0
start_time = time.time()
log_interval = CONFIG.get('log_interval', 100)

pbar = tqdm(dataloader, desc='chesSon Training')
for batch_idx, (boards, move_targets, val_targets) in enumerate(pbar):
    boards       = boards.to(device, non_blocking=True)
    move_targets = move_targets.to(device, non_blocking=True)
    val_targets  = val_targets.unsqueeze(1).to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
        p_logits, v_pred = model(boards)
        loss_p = policy_criterion(p_logits, move_targets)
        loss_v = value_criterion(v_pred, val_targets)
        loss   = loss_p + 0.5 * loss_v

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    with torch.no_grad():
        _, top3 = p_logits.topk(3, dim=1)
        correct_top1   += (top3[:, 0] == move_targets).sum().item()
        correct_top3   += (top3 == move_targets.unsqueeze(1)).any(dim=1).sum().item()
        total_samples  += move_targets.size(0)
        total_loss     += loss.item()

    if (batch_idx + 1) % log_interval == 0:
        pbar.set_postfix({
            'Loss':  f'{total_loss/(batch_idx+1):.3f}',
            'Pol':   f'{loss_p.item():.3f}',
            'Val':   f'{loss_v.item():.3f}',
            'Top1%': f'{100.0*correct_top1/total_samples:.1f}',
            'Top3%': f'{100.0*correct_top3/total_samples:.1f}',
        })

elapsed = time.time() - start_time
print(f'\n✅ Training finished in {elapsed/60:.2f} mins.')
print(f'   Positions: {total_samples:,}')
if total_samples > 0:
    print(f'   Top-1 Accuracy: {100.0*correct_top1/total_samples:.2f}%')
    print(f'   Top-3 Accuracy: {100.0*correct_top3/total_samples:.2f}%')

In [ ]:
# Step 10: Save weights
output_file = CONFIG['output_weights']
torch.save(model.state_dict(), output_file)
print(f'✅ Weights saved to: {output_file}')
print()
print('Next steps:')
print('  1. Download chesson_weights.pt from the Kaggle Output tab')
print('  2. Replace chesSon/chesson/weights/chesson_weights.pt with the downloaded file')
print('  3. Run your engine and enjoy the rating boost! 🎉')